<a href="https://colab.research.google.com/github/ilinashah177/Dissertation-Regressions/blob/main/Copy_of_Regressions_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bond Spread Regression Analysis
================================
Three OLS regression models examining whether biodiversity disclosure and
nature-based solution (NbS) allocations influence green bond spreads.

Variable changes vs original script
-------------------------------------
Dependent variable  : ln(G-Spread)  [was raw G-Spread]
                      Log-transforming a right-skewed spread series improves
                      residual normality and reduces heteroskedasticity.

Maturity control    : ln(TTM)        [was Years_to_Maturity_from_Issue]
                      TTM is measured from a fixed pricing date (2025-03-05),
                      not from issue date. ln(TTM) captures the nonlinear
                      compression of spreads along the yield curve.

Models
------
  Model 1 (Baseline):  ln_Spread ~ DisclosureScore
  Model 2 (+ NbS):     ln_Spread ~ DisclosureScore + NbS
  Model 3 (Full):      ln_Spread ~ DisclosureScore + NbS + ln_TTM + Size

All models use HC3 heteroskedasticity-robust standard errors.
VIF diagnostics are reported for Model 3.

In [1]:
!pip install pandas openpyxl statsmodels

import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings

warnings.filterwarnings("ignore")

In [3]:
FILE_PATH = "Bond_Level_Data_Updated.xlsx"
SHEET     = "Bond_Level_Data"

df = pd.read_excel(FILE_PATH, sheet_name=SHEET)

In [4]:
RENAME = {
    "G-Spread (Mid)"                  : "Spread",
    "Biodiversity Disclosure Score 1-5": "DisclosureScore",
    "NbS_Positive"                    : "NbS",
    "ln Amount"                       : "Size",
    # New columns from Bond_Level_Data_Updated.xlsx
    "TTM\n(years from pricing date)"  : "TTM",
    "ln(TTM)"                         : "ln_TTM",
    "TTM\u00b2"                       : "TTM_sq",
    "ln(G-Spread)"                    : "ln_Spread",
}

df = df.rename(columns=RENAME)

In [5]:
# Drop rows where any regression variable is missing or non-positive
# (ln transforms require strictly positive inputs)

MODEL_VARS = ["ln_Spread", "DisclosureScore", "NbS", "ln_TTM", "Size"]
df_clean   = df[MODEL_VARS].dropna().reset_index(drop=True)

print(f"\n{'='*65}")
print(f"  Sample size: {len(df_clean)} bonds")
print(f"{'='*65}")
print("\nDescriptive Statistics (regression variables):")
print(df_clean.describe().round(4).to_string())


  Sample size: 41 bonds

Descriptive Statistics (regression variables):
       ln_Spread  DisclosureScore      NbS   ln_TTM     Size
count    41.0000          41.0000  41.0000  41.0000  41.0000
mean      3.6903           2.6098   0.5854   1.2327  20.7025
std       1.0008           1.3394   0.4988   0.8732   0.5597
min       0.4847           1.0000   0.0000  -1.3897  20.0198
25%       3.1790           1.0000   0.0000   0.9548  20.2958
50%       3.9330           3.0000   1.0000   1.2335  20.6637
75%       4.2770           4.0000   1.0000   1.7260  20.9278
max       5.4264           5.0000   1.0000   3.1552  22.7411


In [6]:
def run_ols(y, X, label):
    """Fit OLS with HC3 robust SEs and print results."""
    model = sm.OLS(y, sm.add_constant(X)).fit(cov_type="HC3")
    print(f"\n{'='*65}")
    print(f"  {label}")
    print(f"{'='*65}")
    print(model.summary2())
    return model

In [7]:
y = df_clean["ln_Spread"]

X1 = df_clean[["DisclosureScore"]]
X2 = df_clean[["DisclosureScore", "NbS"]]
X3 = df_clean[["DisclosureScore", "NbS", "ln_TTM", "Size"]]

In [8]:
m1 = run_ols(y, X1, "Model 1 (Baseline):  ln_Spread ~ DisclosureScore")


  Model 1 (Baseline):  ln_Spread ~ DisclosureScore
                 Results: Ordinary least squares
Model:              OLS              Adj. R-squared:     -0.007  
Dependent Variable: ln_Spread        AIC:                118.6604
Date:               2026-03-16 23:46 BIC:                122.0875
No. Observations:   41               Log-Likelihood:     -57.330 
Df Model:           1                F-statistic:        0.9486  
Df Residuals:       39               Prob (F-statistic): 0.336   
R-squared:          0.018            Scale:              1.0088  
-----------------------------------------------------------------
                    Coef.  Std.Err.    z    P>|z|   [0.025 0.975]
-----------------------------------------------------------------
const               3.9528   0.2715 14.5585 0.0000  3.4206 4.4849
DisclosureScore    -0.1006   0.1033 -0.9740 0.3301 -0.3030 0.1018
-----------------------------------------------------------------
Omnibus:              17.556       Durbin

In [9]:
m2 = run_ols(y, X2, "Model 2 (+ NbS):     ln_Spread ~ DisclosureScore + NbS")


  Model 2 (+ NbS):     ln_Spread ~ DisclosureScore + NbS
                 Results: Ordinary least squares
Model:              OLS              Adj. R-squared:     0.133   
Dependent Variable: ln_Spread        AIC:                113.4407
Date:               2026-03-16 23:46 BIC:                118.5814
No. Observations:   41               Log-Likelihood:     -53.720 
Df Model:           2                F-statistic:        2.437   
Df Residuals:       38               Prob (F-statistic): 0.101   
R-squared:          0.177            Scale:              0.86815 
-----------------------------------------------------------------
                   Coef.  Std.Err.    z    P>|z|   [0.025  0.975]
-----------------------------------------------------------------
const              3.3674   0.3646  9.2357 0.0000  2.6528  4.0820
DisclosureScore    0.5602   0.2979  1.8803 0.0601 -0.0237  1.1441
NbS               -1.9460   0.9142 -2.1287 0.0333 -3.7377 -0.1542
-----------------------------------

In [10]:
m3 = run_ols(y, X3, "Model 3 (Full):      ln_Spread ~ DisclosureScore + NbS + ln_TTM + Size")


  Model 3 (Full):      ln_Spread ~ DisclosureScore + NbS + ln_TTM + Size
                 Results: Ordinary least squares
Model:              OLS              Adj. R-squared:     0.352   
Dependent Variable: ln_Spread        AIC:                103.3098
Date:               2026-03-16 23:46 BIC:                111.8777
No. Observations:   41               Log-Likelihood:     -46.655 
Df Model:           4                F-statistic:        5.856   
Df Residuals:       36               Prob (F-statistic): 0.000978
R-squared:          0.417            Scale:              0.64923 
-----------------------------------------------------------------
                   Coef.  Std.Err.    z    P>|z|   [0.025  0.975]
-----------------------------------------------------------------
const              9.7702   4.4324  2.2042 0.0275  1.0828 18.4576
DisclosureScore    0.2146   0.3631  0.5911 0.5544 -0.4970  0.9263
NbS               -0.9287   1.0183 -0.9120 0.3618 -2.9246  1.0672
ln_TTM             

In [11]:
print(f"\n{'='*65}")
print("  Variance Inflation Factors – Model 3")
print(f"{'='*65}")

X3_const = sm.add_constant(X3)
vif_df = pd.DataFrame({
    "Variable": X3_const.columns,
    "VIF"     : [variance_inflation_factor(X3_const.values, i)
                 for i in range(X3_const.shape[1])]
})
vif_df = vif_df[vif_df["Variable"] != "const"].reset_index(drop=True)
print(vif_df.to_string(index=False))
print("\n  Rule of thumb: VIF > 10 indicates problematic multicollinearity.")


  Variance Inflation Factors – Model 3
       Variable      VIF
DisclosureScore 7.399414
            NbS 7.511002
         ln_TTM 1.090785
           Size 1.183597

  Rule of thumb: VIF > 10 indicates problematic multicollinearity.


In [12]:
print(f"\n{'='*65}")
print("  Coefficient Comparison Across Models")
print(f"{'='*65}")

coef_table = pd.DataFrame({
    name: model.params for name, model in
    [("Model 1", m1), ("Model 2", m2), ("Model 3", m3)]
}).round(4)
print(coef_table.to_string())

print(f"\n{'='*65}")
print("  Goodness-of-Fit")
print(f"{'='*65}")
for name, model in [("Model 1", m1), ("Model 2", m2), ("Model 3", m3)]:
    n   = int(model.nobs)
    r2  = model.rsquared
    ar2 = model.rsquared_adj
    f   = model.fvalue
    fp  = model.f_pvalue
    print(f"  {name}: N={n}  R²={r2:.4f}  Adj.R²={ar2:.4f}  "
          f"F={f:.3f}  p(F)={fp:.4f}")

print(f"\n{'='*65}")
print("  Notes")
print(f"{'='*65}")
print("  Dependent variable : ln(G-Spread Mid) — natural log of G-spread in bps")
print("  ln_TTM             : ln(time-to-maturity from 2025-03-05 in years)")
print("  Size               : ln(amount issued)")
print("  NbS                : 1 if bond has nature-based solution allocation, else 0")
print("  All SEs            : HC3 heteroskedasticity-robust")
print(f"{'='*65}\n")



  Coefficient Comparison Across Models
                 Model 1  Model 2  Model 3
DisclosureScore  -0.1006   0.5602   0.2146
NbS                  NaN  -1.9460  -0.9287
Size                 NaN      NaN  -0.3275
const             3.9528   3.3674   9.7702
ln_TTM               NaN      NaN   0.5543

  Goodness-of-Fit
  Model 1: N=41  R²=0.0181  Adj.R²=-0.0071  F=0.949  p(F)=0.3361
  Model 2: N=41  R²=0.1766  Adj.R²=0.1333  F=2.437  p(F)=0.1010
  Model 3: N=41  R²=0.4167  Adj.R²=0.3519  F=5.856  p(F)=0.0010

  Notes
  Dependent variable : ln(G-Spread Mid) — natural log of G-spread in bps
  ln_TTM             : ln(time-to-maturity from 2025-03-05 in years)
  Size               : ln(amount issued)
  NbS                : 1 if bond has nature-based solution allocation, else 0
  All SEs            : HC3 heteroskedasticity-robust

